# 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

In [ ]:
# 시각화 설정

!pip install koreanize-matplotlib -q
import koreanize_matplotlib
import matplotlib.pyplot as plt

# EDA — 유저 · 학교 네트워크 (이상치 제거 버전)

**우선순위 2개**
1. 학교별 · 학급별 가입자 수 차이
2. 친구 수가 0명인 사용자는 얼마나 되는가

**실제 스키마 확인 완료**
- `stg_group`: `group_id`(PK), `grade`, `class_num`, `school_id`
- `stg_school`: `school_id`(PK), `address`, `school_type`, `student_count`
- `stg_users`: `user_id`(PK), `signup_at`, `group_id`, `gender`, `block_count`, `hide_count`,
  `friend_count` , `point`, `report_count`, `alarm_count`,
  `pending_chat`, `pending_votes`, `is_push_on`, `ban_status`, `is_valid_user`, `is_staff_account`

**이번 버전의 제외 기준**
아래 항목들을 진단했고, 이 중 **학년(grade) 이상치만 실제로 제외**합니다.
1. 학교 침투율(penetration_pct) 100% 초과 → 진단 결과 테이블 PK 중복은 없어 정상 오차로 판단, **제외하지 않음**
2. 동일 `device_id`에서 여러 `user_id`가 생성된 중복 계정 → **제외하지 않음** (탐지 방식이 event_properties.user_id 캐스팅 성공분(63.6%)에만 의존해 신뢰도가 낮아 이번 버전에서는 보수적으로 포함)
3. 학년(grade) 이상치 — 중학교(M)·고등학교(H) 모두 정상 범위는 1~3학년인데 4학년·20학년 등 존재 확인 → **제외함** (반편성으로 설명 불가한 명백한 오류)

## 0. 이상치 · 중복 계정 진단

아래를 진단합니다. (제외 여부는 항목별로 다름 — 0-6 참고)
1. 학교 침투율(penetration_pct) 100% 초과 현황
2. `student_count`가 비정상적으로 작은 학교
3. 동일 `device_id`에서 여러 `user_id`가 생성된 중복 계정
4. `class_num` 이상치 (참고용, 제외하지 않음)
5. `grade` 이상치 (확정 — 이후 분석에서 제외)

### 0-1. 학교 침투율 100% 초과 현황

In [ ]:
%%bigquery df_over_penetration --project your-gcp-project

SELECT
  s.school_id,
  s.school_type,
  s.student_count,
  COUNT(u.user_id) AS signup_count,
  ROUND(COUNT(u.user_id) / NULLIF(s.student_count, 0) * 100, 1) AS penetration_pct
FROM `your-gcp-project.stage.stg_school` s
LEFT JOIN `your-gcp-project.stage.stg_group` g
  ON s.school_id = g.school_id
LEFT JOIN `your-gcp-project.stage.stg_users` u
  ON g.group_id = u.group_id
GROUP BY s.school_id, s.school_type, s.student_count
HAVING penetration_pct > 100
ORDER BY penetration_pct DESC

In [ ]:
df_over_penetration

In [ ]:
print(f"침투율 100% 초과 학교 수: {len(df_over_penetration)}건")

### 0-2. student_count 규모별 이상치 분포 확인
학생수가 극소(≤20명)인 학교에서 침투율 초과가 집중되는지 확인합니다.

In [ ]:
%%bigquery df_penetration_breakdown --project your-gcp-project

WITH school_signup AS (
  SELECT
    s.school_id, s.school_type, s.student_count,
    COUNT(u.user_id) AS signup_count,
    ROUND(COUNT(u.user_id) / NULLIF(s.student_count, 0) * 100, 1) AS penetration_pct
  FROM `your-gcp-project.stage.stg_school` s
  LEFT JOIN `your-gcp-project.stage.stg_group` g ON s.school_id = g.school_id
  LEFT JOIN `your-gcp-project.stage.stg_users` u ON g.group_id = u.group_id
  GROUP BY s.school_id, s.school_type, s.student_count
)
SELECT
  CASE
    WHEN student_count <= 20 THEN '학생수 20명 이하 (소규모/이상치 의심)'
    ELSE '학생수 21명 이상'
  END AS size_group,
  COUNT(*) AS school_count,
  ROUND(AVG(penetration_pct), 1) AS avg_penetration,
  MAX(penetration_pct) AS max_penetration
FROM school_signup
WHERE penetration_pct > 100
GROUP BY size_group
ORDER BY school_count DESC

In [ ]:
df_penetration_breakdown

### 0-3. 중복 계정 확인 (device_id 기준)
동일 기기에서 여러 user_id가 생성된 경우를 집계합니다. (참고: event_properties.user_id 캐스팅 성공분 63.6%만 대상)

In [ ]:
%%bigquery df_duplicate_check --project your-gcp-project

WITH user_device AS (
  SELECT DISTINCT SAFE_CAST(user_id AS INT64) AS user_id, device_id
  FROM `your-gcp-project.hackle.event_properties`
  WHERE SAFE_CAST(user_id AS INT64) IS NOT NULL
),
device_user_count AS (
  SELECT device_id, COUNT(DISTINCT user_id) AS user_count, MIN(user_id) AS keep_user_id
  FROM user_device
  GROUP BY device_id
)
SELECT
  COUNT(DISTINCT device_id) AS total_devices,
  COUNTIF(user_count > 1) AS duplicate_devices,
  SUM(IF(user_count > 1, user_count - 1, 0)) AS duplicate_user_count
FROM device_user_count

In [ ]:
df_duplicate_check

### 0-4. class_num 이상치 확인 (참고용)
비정상적으로 큰 class_num이 있는지 확인만 합니다. 대형 학교의 정상 반편성일 수 있어, 이번 분석에서는 제외하지 않습니다.

In [ ]:
%%bigquery df_class_num_dist --project your-gcp-project

SELECT
  class_num,
  COUNT(*) AS group_count
FROM `your-gcp-project.stage.stg_group`
GROUP BY class_num
ORDER BY class_num

In [ ]:
df_class_num_dist

### 0-5. 학년(grade) 이상치 확인 (확정 — 제외 대상)
school_type별 정상 학년 범위를 벗어나는 값이 있는지 확인합니다.

In [ ]:
%%bigquery df_school_type_list --project your-gcp-project

SELECT school_type, COUNT(*) AS school_count
FROM `your-gcp-project.stage.stg_school`
GROUP BY school_type
ORDER BY school_type

In [ ]:
df_school_type_list

In [ ]:
%%bigquery df_grade_dist --project your-gcp-project

SELECT
  s.school_type,
  g.grade,
  COUNT(*) AS group_count
FROM `your-gcp-project.stage.stg_group` g
JOIN `your-gcp-project.stage.stg_school` s
  ON g.school_id = s.school_id
GROUP BY s.school_type, g.grade
ORDER BY s.school_type, g.grade

In [ ]:
df_grade_dist

**중학교(M)에서 4학년·20학년 확인됨 → 데이터 오류로 판단, 제외 대상에 추가**

아래 쿼리로 해당 그룹들의 상세 내역(가입자 수 포함)을 확인합니다.

In [ ]:
%%bigquery df_grade_anomaly_detail --project your-gcp-project

SELECT
  g.group_id,
  g.school_id,
  s.school_type,
  g.grade,
  g.class_num,
  COUNT(u.user_id) AS signup_count
FROM `your-gcp-project.stage.stg_group` g
JOIN `your-gcp-project.stage.stg_school` s
  ON g.school_id = s.school_id
LEFT JOIN `your-gcp-project.stage.stg_users` u
  ON g.group_id = u.group_id
WHERE g.grade NOT IN (1, 2, 3)
GROUP BY g.group_id, g.school_id, s.school_type, g.grade, g.class_num
ORDER BY g.grade DESC

In [ ]:
df_grade_anomaly_detail

### 0-6. 잔존 침투율 초과(100~105%) 원인 확인
일부 학교(주로 대형 학교)에서 침투율이 100~105% 수준으로 남아있어, 테이블 자체의 PK 중복 여부를 확인합니다.

In [ ]:
%%bigquery df_users_pk_check --project your-gcp-project

SELECT user_id, COUNT(*) AS row_count
FROM `your-gcp-project.stage.stg_users`
GROUP BY user_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 20

In [ ]:
df_users_pk_check

In [ ]:
%%bigquery df_group_pk_check --project your-gcp-project

SELECT group_id, COUNT(*) AS row_count
FROM `your-gcp-project.stage.stg_group`
GROUP BY group_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 20

In [ ]:
df_group_pk_check

**확인 결과: 두 쿼리 모두 0건 → 테이블 자체 PK 중복 없음**

남은 100~105% 수준의 침투율 초과는 데이터 오류가 아니라, `student_count` 스냅샷 시점과 실제 가입 시점 사이의 자연스러운 오차로 판단합니다.

### 0-7. 이상치 제거 기준 정리 (최종)

- **학교 이상치 (`student_count <= 20`)**: 진단만 하고 **제외하지 않음**
- **중복 계정 (device_id 기준)**: 진단만 하고 **제외하지 않음** (탐지 신뢰도가 낮아 보수적으로 포함 유지)
- **학년(grade) 이상치**: 중학교(M)·고등학교(H) 모두 정상 범위는 1~3학년 → `grade NOT IN (1,2,3)`인 그룹(및 소속 유저) **제외**
- **class_num 이상치**: 진단만 하고 **제외하지 않음** (대형 학교의 정상 반편성일 가능성이 있어 원인 미확정)
- **잔존 침투율 100~105%**: 테이블 PK 중복 없음을 확인, 정상 오차 범위로 간주하고 **제외하지 않음**

아래부터는 **학년 이상치만 제외한** 데이터셋으로 1번(학교/학급별 가입자 수), 2번(친구 수 0명) 분석을 진행합니다.

## 1. 학교별 · 학급별 가입자 수 차이 (학년 이상치 제거 버전)

### 1-1. 학교별 가입자 수 + 침투율 (클린)

In [ ]:
%%bigquery df_school_signup_clean --project your-gcp-project

WITH invalid_groups AS (
  -- 중학교(M)·고등학교(H) 모두 정상 학년은 1~3학년. 그 외 값(예: 4, 20)은 데이터 오류로 간주
  SELECT group_id
  FROM `your-gcp-project.stage.stg_group`
  WHERE grade NOT IN (1, 2, 3)
)
SELECT
  s.school_id,
  s.school_type,
  s.student_count,
  COUNT(u.user_id) AS signup_count,
  ROUND(COUNT(u.user_id) / NULLIF(s.student_count, 0) * 100, 1) AS penetration_pct
FROM `your-gcp-project.stage.stg_school` s
LEFT JOIN `your-gcp-project.stage.stg_group` g
  ON s.school_id = g.school_id
LEFT JOIN `your-gcp-project.stage.stg_users` u
  ON g.group_id = u.group_id
WHERE g.group_id IS NULL OR g.group_id NOT IN (SELECT group_id FROM invalid_groups)
GROUP BY s.school_id, s.school_type, s.student_count
ORDER BY signup_count

In [ ]:
df_school_signup_clean

### 1-2. 학교별 가입자 수 분포 요약 (클린)

In [ ]:
%%bigquery df_school_signup_stats_clean --project your-gcp-project

WITH invalid_groups AS (
  -- 중학교(M)·고등학교(H) 모두 정상 학년은 1~3학년. 그 외 값(예: 4, 20)은 데이터 오류로 간주
  SELECT group_id
  FROM `your-gcp-project.stage.stg_group`
  WHERE grade NOT IN (1, 2, 3)
),
school_signup AS (
  SELECT s.school_id, COUNT(u.user_id) AS signup_count
  FROM `your-gcp-project.stage.stg_school` s
  LEFT JOIN `your-gcp-project.stage.stg_group` g ON s.school_id = g.school_id
  LEFT JOIN `your-gcp-project.stage.stg_users` u ON g.group_id = u.group_id
  WHERE g.group_id IS NULL OR g.group_id NOT IN (SELECT group_id FROM invalid_groups)
  GROUP BY s.school_id
)
SELECT
  COUNT(*) AS total_schools,
  MIN(signup_count) AS min_signup,
  MAX(signup_count) AS max_signup,
  ROUND(AVG(signup_count), 1) AS avg_signup,
  APPROX_QUANTILES(signup_count, 4)[OFFSET(2)] AS median_signup
FROM school_signup

In [ ]:
df_school_signup_stats_clean

### 1-3. 학교별 가입자 수 분포 시각화 (클린)

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(df_school_signup_clean['signup_count'], bins=30)
plt.xlabel('학교별 가입자 수 (이상 학년 그룹 제외)')
plt.ylabel('학교 수')
plt.title('학교별 가입자 수 분포 (클린 버전)')
plt.show()

### 1-4. 학급별 가입자 수 (게이팅 기준 4명과 대조, 클린)
기획서 전제: 학교 40명 이상 · 반 4명 이상이어야 질문 세트가 실행됨.

In [ ]:
%%bigquery df_class_signup_clean --project your-gcp-project

WITH invalid_groups AS (
  -- 중학교(M)·고등학교(H) 모두 정상 학년은 1~3학년. 그 외 값(예: 4, 20)은 데이터 오류로 간주
  SELECT group_id
  FROM `your-gcp-project.stage.stg_group`
  WHERE grade NOT IN (1, 2, 3)
)
SELECT
  g.group_id,
  g.school_id,
  g.grade,
  g.class_num,
  COUNT(u.user_id) AS signup_count
FROM `your-gcp-project.stage.stg_group` g
LEFT JOIN `your-gcp-project.stage.stg_users` u
  ON g.group_id = u.group_id
WHERE g.group_id NOT IN (SELECT group_id FROM invalid_groups)
GROUP BY g.group_id, g.school_id, g.grade, g.class_num
ORDER BY signup_count DESC

In [ ]:
df_class_signup_clean

In [ ]:
%%bigquery df_class_below_threshold_clean --project your-gcp-project

WITH invalid_groups AS (
  -- 중학교(M)·고등학교(H) 모두 정상 학년은 1~3학년. 그 외 값(예: 4, 20)은 데이터 오류로 간주
  SELECT group_id
  FROM `your-gcp-project.stage.stg_group`
  WHERE grade NOT IN (1, 2, 3)
),
class_signup AS (
  SELECT g.group_id, COUNT(u.user_id) AS signup_count
  FROM `your-gcp-project.stage.stg_group` g
  LEFT JOIN `your-gcp-project.stage.stg_users` u ON g.group_id = u.group_id
  WHERE g.group_id NOT IN (SELECT group_id FROM invalid_groups)
  GROUP BY g.group_id
)
SELECT
  COUNTIF(signup_count < 4) AS below_4_count,
  COUNT(*) AS total_class_count,
  ROUND(COUNTIF(signup_count < 4) / COUNT(*) * 100, 1) AS below_4_pct
FROM class_signup

In [ ]:
df_class_below_threshold_clean
# 이상 학년 그룹 제외 후에도 below_4_pct가 원본(32.6%)과 크게 다르면,
# 기존 게이팅 미충족 비율에 데이터 오류 영향이 섞여있었다는 뜻

## 2. 친구 수가 0명인 사용자는 얼마나 되는가 (학년 이상치 제거 버전)

### 2-1. 친구 수 0명 비율 (클린)

In [ ]:
%%bigquery df_zero_friend_clean --project your-gcp-project

WITH invalid_groups AS (
  -- 중학교(M)·고등학교(H) 모두 정상 학년은 1~3학년. 그 외 값(예: 4, 20)은 데이터 오류로 간주
  SELECT group_id
  FROM `your-gcp-project.stage.stg_group`
  WHERE grade NOT IN (1, 2, 3)
)
SELECT
  COUNTIF(friend_count = 0) AS zero_friend_count,
  COUNTIF(friend_count > 0) AS has_friend_count,
  COUNT(*) AS total_users,
  ROUND(COUNTIF(friend_count = 0) / COUNT(*) * 100, 1) AS zero_friend_pct
FROM `your-gcp-project.stage.stg_users` u
LEFT JOIN `your-gcp-project.stage.stg_group` g ON u.group_id = g.group_id
WHERE g.group_id IS NULL OR g.group_id NOT IN (SELECT group_id FROM invalid_groups)

In [ ]:
df_zero_friend_clean

### 2-2. 친구 수 분포 (클린)

In [ ]:
%%bigquery df_friend_dist_clean --project your-gcp-project

WITH invalid_groups AS (
  -- 중학교(M)·고등학교(H) 모두 정상 학년은 1~3학년. 그 외 값(예: 4, 20)은 데이터 오류로 간주
  SELECT group_id
  FROM `your-gcp-project.stage.stg_group`
  WHERE grade NOT IN (1, 2, 3)
)
SELECT
  u.friend_count,
  COUNT(*) AS user_count
FROM `your-gcp-project.stage.stg_users` u
LEFT JOIN `your-gcp-project.stage.stg_group` g ON u.group_id = g.group_id
WHERE g.group_id IS NULL OR g.group_id NOT IN (SELECT group_id FROM invalid_groups)
GROUP BY u.friend_count
ORDER BY u.friend_count
LIMIT 30

In [ ]:
df_friend_dist_clean

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(df_friend_dist_clean['friend_count'], df_friend_dist_clean['user_count'])
plt.xlabel('친구 수')
plt.ylabel('유저 수')
plt.title('유저별 친구 수 분포 (상위 30구간, 클린 버전)')
plt.show()

### 2-3. 친구 0명 비율이 학급 규모와 관계있는지 (클린)

In [ ]:
%%bigquery df_zero_friend_by_class_size_clean --project your-gcp-project

WITH invalid_groups AS (
  -- 중학교(M)·고등학교(H) 모두 정상 학년은 1~3학년. 그 외 값(예: 4, 20)은 데이터 오류로 간주
  SELECT group_id
  FROM `your-gcp-project.stage.stg_group`
  WHERE grade NOT IN (1, 2, 3)
),
class_size AS (
  SELECT g.group_id, COUNT(u.user_id) AS class_signup_count
  FROM `your-gcp-project.stage.stg_group` g
  LEFT JOIN `your-gcp-project.stage.stg_users` u ON g.group_id = u.group_id
  WHERE g.group_id NOT IN (SELECT group_id FROM invalid_groups)
  GROUP BY g.group_id
)
SELECT
  CASE
    WHEN cs.class_signup_count < 4 THEN '4명 미만 (게이팅 미충족)'
    WHEN cs.class_signup_count < 10 THEN '4~9명'
    WHEN cs.class_signup_count < 20 THEN '10~19명'
    ELSE '20명 이상'
  END AS class_size_bucket,
  COUNT(u.user_id) AS total_users,
  COUNTIF(u.friend_count = 0) AS zero_friend_users,
  ROUND(COUNTIF(u.friend_count = 0) / COUNT(u.user_id) * 100, 1) AS zero_friend_pct
FROM `your-gcp-project.stage.stg_users` u
JOIN `your-gcp-project.stage.stg_group` g ON u.group_id = g.group_id
JOIN class_size cs ON g.group_id = cs.group_id
WHERE g.group_id NOT IN (SELECT group_id FROM invalid_groups)
GROUP BY class_size_bucket
ORDER BY class_size_bucket

In [ ]:
df_zero_friend_by_class_size_clean

### 2-4. 테스트/무효 계정 + 이상 학년 모두 제외한 버전

#### 테스트/무효 계정

In [ ]:
%%bigquery df_account_status_breakdown --project your-gcp-project

SELECT
  is_valid_user,
  is_staff_account,
  ban_status,
  COUNT(*) AS user_count,
  COUNTIF(friend_count = 0) AS zero_friend_count,
  ROUND(COUNTIF(friend_count = 0) / COUNT(*) * 100, 1) AS zero_friend_pct
FROM `your-gcp-project.stage.stg_users`
GROUP BY is_valid_user, is_staff_account, ban_status
ORDER BY user_count DESC

In [ ]:
df_account_status_breakdown

#### 테스트 계정의 학년 확인

In [ ]:
%%bigquery df_staff_account_detail --project your-gcp-project

SELECT
  u.user_id,
  u.group_id,
  g.school_id,
  s.school_type,
  g.grade,
  g.class_num,
  u.friend_count,
  u.is_valid_user,
  u.is_staff_account,
  u.ban_status
FROM `your-gcp-project.stage.stg_users` u
LEFT JOIN `your-gcp-project.stage.stg_group` g
  ON u.group_id = g.group_id
LEFT JOIN `your-gcp-project.stage.stg_school` s
  ON g.school_id = s.school_id
WHERE u.is_valid_user = TRUE
  AND u.is_staff_account = TRUE
  AND u.ban_status = 'N'

In [ ]:
df_staff_account_detail

In [ ]:
%%bigquery df_zero_friend_valid_only_clean --project your-gcp-project

WITH invalid_groups AS (
  -- 중학교(M)·고등학교(H) 모두 정상 학년은 1~3학년. 그 외 값(예: 4, 20)은 데이터 오류로 간주
  SELECT group_id
  FROM `your-gcp-project.stage.stg_group`
  WHERE grade NOT IN (1, 2, 3)
)
SELECT
  COUNTIF(friend_count = 0) AS zero_friend_count,
  COUNT(*) AS total_users,
  ROUND(COUNTIF(friend_count = 0) / COUNT(*) * 100, 1) AS zero_friend_pct
FROM `your-gcp-project.stage.stg_users` u
LEFT JOIN `your-gcp-project.stage.stg_group` g ON u.group_id = g.group_id
WHERE u.is_valid_user = TRUE
  AND u.is_staff_account = FALSE
  AND (g.group_id IS NULL OR g.group_id NOT IN (SELECT group_id FROM invalid_groups))

In [ ]:
df_zero_friend_valid_only_clean

In [ ]:
%%bigquery df_grade_anomaly_user_status --project your-gcp-project

SELECT
  u.user_id,
  u.is_valid_user,
  u.is_staff_account,
  u.ban_status,
  g.grade,
  g.class_num
FROM `your-gcp-project.stage.stg_users` u
JOIN `your-gcp-project.stage.stg_group` g
  ON u.group_id = g.group_id
WHERE g.grade NOT IN (1, 2, 3)

In [ ]:
df_grade_anomaly_user_status